In [15]:
import torch
import librosa
import torchaudio 
import numpy as np
import pandas as pd
import whisper
import spacy
import json
import re
import subprocess
from pyannote.audio import Pipeline
from transformers import WavLMForSequenceClassification, Wav2Vec2FeatureExtractor
from pydub import AudioSegment
import os

# --- 0. CONFIGURACIÓN DE DISPOSITIVO (SOLUCIÓN AL ERROR) ---
# Creamos un objeto torch.device en lugar de usar un string "cuda"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Trabajando en: {device}")

# --- 1. CARGA DE MODELOS ---
try:
    nlp = spacy.load("en_core_web_sm") 
except:
    print("Descargando modelo de spaCy...")
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Configuración de Rutas
NOTEBOOK_DIR = os.getcwd()
MODEL_PATH = os.path.join(NOTEBOOK_DIR, "RAVDESS_WavLM_Final")
# Asegúrate de que este archivo existe o cámbiale el nombre al tuyo
VIDEO_PATH = os.path.join(NOTEBOOK_DIR, "../solucion_1_audio_prompt/input/starwars.mp3") 
HF_TOKEN = "TU_TOKEN_AQUI" # Reemplaza con tu token real

print("Cargando Modelos de IA...")
# Modelo de Emoción corregido con .to(device)
emotion_model = WavLMForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    use_safetensors=True
).to(device)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)
whisper_model = whisper.load_model("base")

# Pipeline de Diarización corregido con .to(device)
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", 
    token=HF_TOKEN
)
diarization_pipeline.to(device)

# --- 2. LÓGICA DE AUTO-IDENTIFICACIÓN DINÁMICA ---

def auto_identify_speakers(transcription_segments, diarization_result):
    speaker_guesses = {}
    
    for turn, _, speaker_id in diarization_result.itertracks(yield_label=True):
        text = ""
        for seg in transcription_segments:
            if (seg['start'] <= turn.end) and (seg['end'] >= turn.start):
                text += " " + seg['text']
        
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                name = ent.text.strip().upper()
                if speaker_id not in speaker_guesses: speaker_guesses[speaker_id] = {}
                speaker_guesses[speaker_id][name] = speaker_guesses[speaker_id].get(name, 0) + 1

    final_mapping = {}
    used_names = set()
    
    for sid, names in speaker_guesses.items():
        if names:
            sorted_names = sorted(names.items(), key=lambda x: x[1], reverse=True)
            for name, freq in sorted_names:
                if name not in used_names:
                    final_mapping[sid] = name
                    used_names.add(name)
                    break
        if sid not in final_mapping:
            final_mapping[sid] = sid 

    return final_mapping

# --- 3. PROCESAMIENTO PRINCIPAL ---

def analyze_any_audio(file_path):
    # Usar ruta absoluta para evitar confusiones de carpetas en Windows
    audio_path = os.path.abspath("temp_audio.wav")
    file_path_abs = os.path.abspath(file_path)

    print(f"Convirtiendo {file_path_abs} a WAV...")
    
    # Comando robusto con subprocess (maneja mejor las comillas y espacios)
    command = [
        "ffmpeg", "-i", file_path_abs,
        "-ac", "1", "-ar", "16000",
        audio_path, "-y"
    ]
    
    try:
        # Ejecutamos y esperamos a que termine
        result = subprocess.run(command, capture_output=True, text=True, check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"❌ Error en FFmpeg: {e.stderr}")

    # VERIFICACIÓN CRUCIAL: ¿Existe el archivo realmente?
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"❌ FFmpeg no generó el archivo en: {audio_path}")

    # A. Diarización y Transcripción
    print("Cargando audio en memoria...")
    waveform, sr = torchaudio.load(audio_path)
    
    print("Identificando hablantes (Diarización)...")
    diarization = diarization_pipeline({"waveform": waveform, "sample_rate": sr})
    
    print("Transcribiendo audio (Whisper)...")
    transcription = whisper_model.transcribe(audio_path)

    # B. Auto-Identificar quién es quién
    print("Identificando personajes dinámicamente...")
    dynamic_mapping = auto_identify_speakers(transcription['segments'], diarization.speaker_diarization)

    # C. Análisis de Emociones y Menciones
    full_audio = AudioSegment.from_wav(audio_path)
    results = []

    print("Procesando segmentos emocionales...")
    for turn, _, speaker_id in diarization.speaker_diarization.itertracks(yield_label=True):
        if (turn.end - turn.start) < 0.5: continue

        speaker_name = dynamic_mapping.get(speaker_id, speaker_id)

        # Cortar audio
        seg_audio = full_audio[int(turn.start*1000):int(turn.end*1000)]
        samples = np.array(seg_audio.get_array_of_samples()).astype(np.float32) / 32768.0
        
        inputs = feature_extractor(
            samples, 
            sampling_rate=16000, 
            padding=True, 
            do_normalize=True, 
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            logits = emotion_model(**inputs).logits
            emotion = emotion_model.config.id2label[torch.argmax(logits, dim=-1).item()]

        # Texto y Menciones
        best_text = ""
        for seg in transcription['segments']:
            overlap = min(turn.end, seg['end']) - max(turn.start, seg['start'])
            if overlap > 0.1: # Al menos 0.1s de coincidencia
                best_text = seg['text'].strip()
                break

        doc = nlp(best_text)
        mentions = list(set([ent.text.upper() for ent in doc.ents if ent.label_ == "PERSON" and ent.text.upper() != speaker_name]))

        results.append({
            "start": round(turn.start, 2),
            "end": round(turn.end, 2),
            "speaker": speaker_name,
            "emotion": emotion,
            "mentions": mentions,
            "text": best_text
        })

    # Guardar JSON
    output_json = "dynamic_scene_analysis.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    
    # Limpieza: Intentar borrar con manejo de errores por si Windows lo bloquea
    try:
        if os.path.exists(audio_path):
            os.remove(audio_path)
    except:
        print(f"Aviso: No se pudo borrar {audio_path}, puede estar en uso.")

    return results

# --- EJECUCIÓN ---
try:
    results = analyze_any_audio(VIDEO_PATH)
    print(f"✅ ¡Éxito! Se ha generado '{os.getcwd()}/dynamic_scene_analysis.json'")
    # Mostrar los primeros resultados
    for r in results[:5]:
        print(f"[{r['start']}s] {r['speaker']} ({r['emotion']}): {r['text'][:50]}...")
except Exception as e:
    print(f"❌ Error durante la ejecución: {e}")

Trabajando en: cpu
Cargando Modelos de IA...
Convirtiendo d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_1_audio_prompt\input\starwars.mp3 a WAV...
Cargando audio en memoria...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


Identificando hablantes (Diarización)...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


Transcribiendo audio (Whisper)...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Identificando personajes dinámicamente...
Procesando segmentos emocionales...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


✅ ¡Éxito! Se ha generado 'd:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning/dynamic_scene_analysis.json'
[2.58s] AMADALA (surprise): I don't know how much longer I can hold off the ve...
[6.48s] AMADALA (angry): I don't know how much longer I can hold off the ve...
[16.94s] AMADALA (surprise): If they do break away, I will not let this republi...
[18.54s] AMADALA (angry): If they do break away, I will not let this republi...
[20.11s] AMADALA (surprise): If they do, you must realize there aren't enough J...


In [16]:
import networkx as nx
import matplotlib.pyplot as plt
import imageio
import os
import pandas as pd

# 1. Mapa de colores (consistente con el modelo)
color_map = {
    'neutral': 'grey', 'happy': 'green', 'sad': 'blue',
    'angry': 'red', 'fear': 'purple', 'surprise': 'orange', 'disgust': 'brown'
}

def create_interaction_gif(results_list, output_gif="social_emotion_graph.gif"):
    # Convertimos la lista de resultados en DataFrame si no lo es ya
    df = pd.DataFrame(results_list)
    
    # 2. Identificar todos los nodos (Personajes que hablan + Personajes mencionados)
    all_speakers = set(df['speaker'].unique())
    all_mentions = set([m for sublist in df['mentions'] for m in sublist])
    all_nodes = list(all_speakers.union(all_mentions))
    
    # 3. Crear el Grafo Base
    G = nx.DiGraph()
    G.add_nodes_from(all_nodes)
    
    # Posición fija circular para que el GIF no "salte"
    pos = nx.circular_layout(G) 

    filenames = []
    print(f"Generando frames para {len(df)} segmentos...")

    for i, row in df.iterrows():
        plt.figure(figsize=(10, 8))
        
        current_speaker = row['speaker']
        current_emotion = row['emotion']
        mentions = row['mentions'] # Esto ya es una lista: ['OBI-WAN']
        
        # Color de la emoción actual
        current_color = color_map.get(current_emotion, 'black')

        # --- DIBUJAR NODOS ---
        # Resaltamos al hablante actual con un color diferente o tamaño mayor
        node_colors = ['gold' if node == current_speaker else 'lightgray' for node in G.nodes]
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black', linewidths=2)
        nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

        # --- DIBUJAR FLECHAS (INTERACCIONES) ---
        if mentions:
            for target in mentions:
                if target in G.nodes:
                    # Dibujamos la flecha de la interacción actual
                    nx.draw_networkx_edges(
                        G, pos, edgelist=[(current_speaker, target)],
                        edge_color=current_color,
                        width=6, # Flecha gruesa para la acción actual
                        arrowsize=35, 
                        connectionstyle="arc3,rad=0.2"
                    )
        else:
            # Si no menciona a nadie, podemos dibujar un borde al nodo del hablante
            # para indicar que está "emitiendo" esa emoción al aire/general.
            nx.draw_networkx_nodes(G, pos, nodelist=[current_speaker], 
                                 node_size=3500, node_color='gold', 
                                 edgecolors=current_color, linewidths=5)

        # --- TÍTULO Y TEXTO ---
        # Cortamos el texto si es muy largo para que quepa en el título
        display_text = (row['text'][:60] + '...') if len(row['text']) > 60 else row['text']
        
        plt.title(f"TIEMPO: {row['start']}s\n{current_speaker} ({current_emotion.upper()})\n\"{display_text}\"", 
                  fontsize=12, fontweight='bold', pad=20)
        
        plt.axis('off')
        
        # Guardar frame temporal con numeración formateada (001, 002...)
        filename = f"int_frame_{i:03d}.png"
        plt.savefig(filename, dpi=100, bbox_inches='tight')
        plt.close()
        filenames.append(filename)

    # 4. COMPILAR EL GIF
    print(f"Compilando GIF final: {output_gif}...")
    with imageio.get_writer(output_gif, mode='I', duration=1.0, loop=0) as writer:
        for filename in filenames:
            image = imageio.imread(filename)
            writer.append_data(image)
            # Borrar archivo temporal inmediatamente después de añadirlo al GIF
            os.remove(filename)

    print(f"✅ ¡Proceso completado! GIF guardado como: {output_gif}")

# --- MODO DE USO ---
# Solo tienes que pasarle la variable 'results' que devuelve tu función 'analyze_any_audio'
create_interaction_gif(results)

Generando frames para 51 segmentos...
Compilando GIF final: social_emotion_graph.gif...


C:\Users\pedro\AppData\Local\Temp\ipykernel_9744\763478211.py:86: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)


✅ ¡Proceso completado! GIF guardado como: social_emotion_graph.gif
